## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Large Language Models (LLMs) Vanilla
*****

__Implementación cortesia de:__ [Madhuranjan kumar ](https://github.com/madhuranjank6/ResumeGPT#)

## Librerias

In [ ]:
from urllib.request import urlretrieve
import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from tensorflow.keras import layers


## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model

  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

In [ ]:
def create_training_sequences(sequence, context_window=64):
    """
    Create input-output pairs for training the language model.
    Each input is a sequence of tokens, and the output is the same sequence shifted by one position.

    Args:
        sequence: List of token IDs
        context_window: Size of the context window (sequence length)

    Returns:
        Tuple of (inputs, labels) as numpy arrays
    """
    inputs, labels = [], []

    for i in range(len(sequence) - context_window):

        # Input: tokens i to i+context_window
        input_seq = sequence[i:i+context_window]

        # Label: tokens i+1 to i+context_window+1 (next token prediction)
        label_seq = sequence[i+1:i+context_window+1]

        inputs.append(input_seq)
        labels.append(label_seq)

    return np.array(inputs), np.array(labels)

## Diseño del modelo

In [ ]:
## Customized embedding layer

class TokenAndPositionEmbedding(layers.Layer):
    """
        Two seperate embedding layers, one for tokens, one for token index (positions).
    """
    def __init__(self, maxlen, vocab_size, embed_dim):
        """
            INPUT:
                @param maxlen: sequence length
                @type maxlen: int

                @param vocab_size: maximum number of words to keep (for tokens)
                @type vocab_size: int

                @param embed_dim: dimension of the dense embedding
                @type embed_dim: int
        """
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_embedding = layers.Embedding(input_dim=vocab_size,
                                                output_dim=embed_dim)
        self.position_embedding = layers.Embedding(input_dim=maxlen,
                                                   output_dim=embed_dim)

    def call(self, x):
        """
            INPUT:
                @param x: token sequence
                @type x: numpy.ndarray

            OUTPUT:
                @param results: sum of embeddings
                @type results: tensorflow.tensor
        """

        ## Compute sequence length
        maxlen = tf.shape(x)[-1]

        ## Compute embedding to positions
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.position_embedding(positions)

        ## Compute embedding to sequence
        x = self.token_embedding(x)

        ## Sum of embedding results
        results = x + positions

        ## return results
        return results


In [ ]:
def create_transformer_block(embedding_dim, num_heads, ff_dim, dropout_rate=0.1):
    """
    Create a transformer decoder block with self-attention and feed-forward layers.

    Args:
        embedding_dim: Dimension of the embedding space
        num_heads: Number of attention heads
        ff_dim: Hidden dimension of the feed-forward network
        dropout_rate: Dropout rate for regularization

    Returns:
        A Keras Model representing a transformer block
    """
    inputs = layers.Input(shape=(None, embedding_dim))

    # Self-attention layer
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embedding_dim
    )(inputs, inputs)
    attention_output = layers.Dropout(dropout_rate)(attention_output)

    # First residual connection and normalization
    first_output = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output)

    # Feed-forward network
    ffn = tf.keras.Sequential([
        layers.Dense(ff_dim, activation='relu'),
        layers.Dense(embedding_dim)
    ])
    ffn_output = ffn(first_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)

    # Second residual connection and normalization
    second_output = layers.LayerNormalization(epsilon=1e-6)(first_output + ffn_output)

    return tf.keras.Model(inputs=inputs, outputs=second_output)

In [ ]:
def build_language_model(
    vocab_size=1200,
    max_seq_length=64,
    embedding_dim=96,
    num_heads=2,
    ff_dim=384,
    num_transformer_blocks=2,
    dropout_rate=0.25
):
    """
    Build a decoder-only transformer model for language modeling.

    Args:
        vocab_size: Size of the vocabulary
        max_seq_length: Maximum sequence length
        embedding_dim: Dimension of token embeddings
        num_heads: Number of attention heads
        ff_dim: Feed-forward network hidden dimension
        num_transformer_blocks: Number of transformer blocks to stack
        dropout_rate: Dropout rate for regularization

    Returns:
        A Keras Model representing the language model
    """
    # Input layer
    inputs = layers.Input(shape=(max_seq_length,))

    ## Apply enmbedding
    x = TokenAndPositionEmbedding(max_seq_length, vocab_size, embedding_dim)(inputs)

    # Stack transformer blocks
    for _ in range(num_transformer_blocks):
        x = create_transformer_block(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate
        )(x)

    # Output layer with an intermediate layer
    x = layers.Dense(embedding_dim//2, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(vocab_size, activation='softmax')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='LLM_Vanilla')

In [ ]:
def generate_resume_text(
    seed_text,
    model,
    tokenizer,
    max_seq_length=64,
    num_tokens=50,
    temperature=1.0
):
    """
    Generate text continuation from a seed text using the trained model.

    Args:
        seed_text: Starting text for generation
        model: Trained language model
        tokenizer: Tokenizer used for training
        max_seq_length: Maximum sequence length for model input
        num_tokens: Number of tokens to generate
        temperature: Controls randomness (lower = more deterministic)

    Returns:
        Generated text continuation
    """
    generated_text = seed_text

    for _ in range(num_tokens):
        # Tokenize the current text
        token_sequence = tokenizer.texts_to_sequences([generated_text])[0]

        # Pad or truncate to match model input size
        if len(token_sequence) > max_seq_length:
            # Take only the last max_seq_length tokens
            token_sequence = token_sequence[-max_seq_length:]

        # Pad sequence if needed
        padded_sequence = tf.keras.preprocessing.sequence.pad_sequences(
            [token_sequence],
            maxlen=max_seq_length
        )

        # Get model predictions
        predictions = model.predict(padded_sequence, verbose=0)[0, -1]

        # Apply temperature scaling for controlling randomness
        predictions = np.asarray(predictions).astype('float64')
        predictions = np.log(predictions + 1e-9) / temperature
        predictions = np.exp(predictions) / np.sum(np.exp(predictions))

        # Sample next token based on probabilities
        next_token_id = np.random.choice(len(predictions), p=predictions)
        next_word = tokenizer.index_word.get(next_token_id, '')

        # Add the next word to the generated text
        if next_word:
            generated_text += ' ' + next_word
        else:
            # Break if we get an empty token
            break

    return generated_text

## Dataset

In [ ]:
## download file
urlretrieve('https://raw.githubusercontent.com/adoc-box/Datasets/refs/heads/main/Celulares%202026%20S2.txt', 'data.txt')

# read data
with open("data.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

# Create and fit a tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=5000,  # Vocabulary size
    oov_token="<OOV>"  # Out-of-vocabulary token
)
tokenizer.fit_on_texts([resume_text])

# Convert text to sequence of token IDs
token_sequence = tokenizer.texts_to_sequences([resume_text])[0]
print(f"Sequence length: {len(token_sequence)}")

In [ ]:
print(resume_text)

In [ ]:
# Create training data
input_sequences, target_sequences = create_training_sequences(token_sequence)
print(f"Created {len(input_sequences)} training examples")

## Modelo

In [ ]:
# Model hyperparameters - optimized for small data
VOCAB_SIZE = 1200
MAX_SEQ_LENGTH = 64
EMBEDDING_DIM = 96
NUM_HEADS = 2
FFN_DIM = 384
NUM_TRANSFORMER_BLOCKS = 2
BATCH_SIZE = 8
EPOCHS = 100
DROPOUT_RATE = 0.25

# Build and compile the model
model = build_language_model(
    vocab_size=VOCAB_SIZE,
    max_seq_length=MAX_SEQ_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FFN_DIM,
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    dropout_rate=DROPOUT_RATE
)

# Use weight decay in the optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, weight_decay=0.01)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

In [ ]:
# Train the model
training_history = model.fit(
    input_sequences,
    target_sequences,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1  # Use 10% of data for validation
)

# Save the trained model
#model.save("LLM_Vanilla.h5")

In [ ]:
plot_history(training_history)

In [ ]:
# Generate text with different temperature settings
seed_phrases = [
    "Mi celular favorito es",
    "Fold es",
    "Samsung es mejor que iPhone porque"
]

print("\n==== Generated Resume Text Samples ====\n")
for seed in seed_phrases:
    print(f"Seed: '{seed}'")

    # Generate with different temperature values
    for temp in [0.7, 1.0, 1.3]:
        generated = generate_resume_text(
            seed,
            model,
            tokenizer,
            temperature=temp
        )
        print(f"  Temperature {temp}: {generated}")
    print()